# 阶段1：自动下载数据集 + 数据预处理

TMDB 5000 Movie Dataset

本Notebook完成以下任务：
1. 自动下载TMDB 5000电影数据集
2. 数据合并
3. 数据清洗
4. 特征提取
5. 保存处理后的数据

In [ ]:
# 导入所需库
import requests
import pandas as pd
import json
import time
from datetime import datetime

## 1. 自动下载数据集

In [ ]:
# 定义下载函数，支持自动重试
def download_file(url, save_path, max_retries=3):
    """下载文件，失败自动重试max_retries次"""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            with open(save_path, 'wb') as f:
                f.write(response.content)
            print(f"下载成功: {save_path}")
            return True
        except Exception as e:
            print(f"第{attempt+1}次下载失败: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"等待{wait_time}秒后重试...")
                time.sleep(wait_time)
    return False

In [ ]:
# 数据下载URL和保存路径
movies_url = "https://raw.githubusercontent.com/missvicki/TMDb-movie-data/master/tmdb_5000_movies.csv"
credits_url = "https://raw.githubusercontent.com/missvicki/TMDb-movie-data/master/tmdb_5000_credits.csv"

movies_path = r"D:\电影分析项目\原始数据\tmdb_5000_movies.csv"
credits_path = r"D:\电影分析项目\原始数据\tmdb_5000_credits.csv"

print("开始下载数据集...")
download_file(movies_url, movies_path)
download_file(credits_url, credits_path)
print("数据集下载成功")

## 2. 数据合并

In [ ]:
# 读取两个CSV文件
df_movies = pd.read_csv(movies_path)
df_credits = pd.read_csv(credits_path)

print(f"电影基本信息表行数: {df_movies.shape[0]}, 列数: {df_movies.shape[1]}")
print(f"演职人员信息表行数: {df_credits.shape[0]}, 列数: {df_credits.shape[1]}")

In [ ]:
# 通过movie_id进行内连接合并
df = pd.merge(df_movies, df_credits, left_on='id', right_on='movie_id', how='inner')
df.rename(columns={'title_x': 'title'}, inplace=True)
df.drop(columns=['title_y'], inplace=True)
print(f"合并后总行数: {df.shape[0]}")

## 3. 数据清洗

In [ ]:
# 3.1 删除完全重复的行
before_drop = df.shape[0]
df = df.drop_duplicates()
after_drop = df.shape[0]
print(f"删除重复行: {before_drop - after_drop}行")

In [ ]:
# 3.2 填充文本字段的缺失值
text_fields = ['overview', 'genres', 'cast', 'crew', 'keywords']
for field in text_fields:
    df[field] = df[field].fillna('')

In [ ]:
# 3.3 过滤预算和票房为0或负数的无效数据
before_filter = df.shape[0]
df = df[~((df['budget'] <= 0) | (df['revenue'] <= 0))]
after_filter = df.shape[0]
print(f"过滤budget/revenue<=0: {before_filter - after_filter}行")

In [ ]:
# 3.4 处理日期字段，提取年份和月份
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.astype('Int64')
df['release_month'] = df['release_date'].dt.month.astype('Int64')

In [ ]:
# 3.5 过滤runtime异常值
df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce')
before_runtime = df.shape[0]
df = df[(df['runtime'] >= 60) & (df['runtime'] <= 300)]
after_runtime = df.shape[0]
print(f"过滤runtime异常: {before_runtime - after_runtime}行")

df = df.reset_index(drop=True)
print(f"清洗后数据行数: {df.shape[0]}")

## 4. 特征提取（核心）

In [ ]:
# 4.1 从genres JSON中提取电影类型名称
def extract_genres(genres_str):
    try:
        genres_list = json.loads(genres_str)
        return ' '.join([g['name'] for g in genres_list])
    except:
        return ''

In [ ]:
# 4.2 从cast JSON中提取前3位主要演员
def extract_cast(cast_str, top_n=3):
    try:
        cast_list = json.loads(cast_str)
        return ' '.join([c['name'] for c in cast_list[:top_n]])
    except:
        return ''

In [ ]:
# 4.3 从crew JSON中提取导演信息
def extract_director(crew_str):
    try:
        crew_list = json.loads(crew_str)
        directors = [c['name'] for c in crew_list if c.get('job') == 'Director']
        return ' '.join(directors) if directors else ''
    except:
        return ''

In [ ]:
# 4.4 从keywords JSON中提取关键词
def extract_keywords(keywords_str):
    try:
        keywords_list = json.loads(keywords_str)
        return ' '.join([k['name'] for k in keywords_list])
    except:
        return ''

In [ ]:
# 4.5 批量执行特征提取
df['genres'] = df['genres'].apply(extract_genres)
df['cast'] = df['cast'].apply(lambda x: extract_cast(x, 3))
df['director'] = df['crew'].apply(extract_director)
df['keywords'] = df['keywords'].apply(extract_keywords)

# 删除原始的crew字段
df.drop(columns=['crew'], inplace=True)

In [ ]:
# 4.6 创建combined_features字段
df['combined_features'] = (
    df['overview'].fillna('') + ' ' +
    df['genres'] + ' ' +
    df['cast'] + ' ' +
    df['director'] + ' ' +
    df['keywords']
)

print("特征提取完成")

## 5. 保存结果

In [ ]:
# 选择保留的字段
output_columns = [
    'movie_id', 'title', 'release_year', 'release_month', 'runtime',
    'budget', 'revenue', 'vote_average', 'vote_count', 'original_language',
    'genres', 'director', 'cast', 'keywords', 'overview', 'combined_features'
]

df_final = df[output_columns].copy()
output_path = r"D:\电影分析项目\处理后数据\processed_movies.csv"
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"数据已保存至: {output_path}")

In [ ]:
# 查看最终数据
print(f"最终数据集基本信息:")
print(f"行数: {df_final.shape[0]}, 列数: {df_final.shape[1]}")
print(f"\n字段列表: {list(df_final.columns)}")
print(f"\n前5行数据:")
print(df_final.head())

---
**阶段1完成！** 已生成以下文件：
- 原始数据/tmdb_5000_movies.csv
- 原始数据/tmdb_5000_credits.csv
- 处理后数据/processed_movies.csv